In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from typing import Dict

from transformers import AutoConfig, MllamaConfig

/nethome/lukar/code/squashed-llama/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### LLama 3.2 multi-modal schematic

<img src="20241007-llama3_schematic.png" width=1024 />

In [3]:
def vision_perf(config: MllamaConfig, n_tiles: int) -> Dict[str, int]:
    v_config, t_config = config.vision_config, config.text_config

    # NOTE: n_tiles = {1, 2, 3, 4}
    conv_params = v_config.patch_size**2 * v_config.num_channels * v_config.hidden_size
    conv_flops = (
        2
        * n_tiles
        * v_config.image_size**2
        * v_config.num_channels
        * v_config.hidden_size
    )
    # TODO: Are tiles just flattened, or are they independently passed through self-attn?
    seq_len = (
        n_tiles * (v_config.image_size // v_config.patch_size) ** 2
    )  # NOTE: actually a bit bigger due to padding
    n_layers = v_config.num_hidden_layers + v_config.num_global_layers

    mlp_params = n_layers * 12 * v_config.hidden_size**2
    mlp_flops = 2 * seq_len * mlp_params

    # Final projection to LM hidden_size
    mlp_params += v_config.vision_output_dim * t_config.hidden_size
    mlp_flops += 2 * seq_len * v_config.vision_output_dim * t_config.hidden_size

    attn_flops = (
        2
        * seq_len
        * n_layers
        * 2
        * seq_len
        * v_config.attention_heads
        * (v_config.hidden_size // v_config.attention_heads)
    )

    return dict(
        params=conv_params + mlp_params,
        conv_flops=conv_flops,
        mlp_flops=mlp_flops,
        attn_flops=attn_flops,
        seq_len=seq_len,
    )

In [4]:
def text_perf(
    config: MllamaConfig, prefill_length: int, n_generated: int, img_seq_len: int
) -> Dict[str, int]:
    t_config = config.text_config

    n_layers = t_config.num_hidden_layers  # total number of layers
    n_cross_attn_layers = len(t_config.cross_attention_layers)
    n_attn_layers = n_layers - n_cross_attn_layers

    # (FFN w SwiGLU) - 3 * 3.5 * d^2, (Attn w GQA) - 2.5 * d^2
    mlp_params = n_layers * 13 * t_config.hidden_size**2

    # Per token numbers
    head_dim = t_config.hidden_size // t_config.num_attention_heads

    # Flops per token, seq len 1
    attn_flops_per_tok = 2 * n_attn_layers * 2 * t_config.num_attention_heads * head_dim

    # Flops per one text token
    cross_attn_flops = (
        2
        * n_cross_attn_layers
        * 2
        * img_seq_len
        * t_config.num_attention_heads
        * head_dim
    )

    n_kv_els_per_tok = 2 * t_config.num_key_value_heads * head_dim

    # Average sequence length during generation
    avg_seq_len = prefill_length + n_generated / 2

    # Embedding
    embed_params = 2 * t_config.hidden_size * t_config.vocab_size

    return dict(
        params=mlp_params + embed_params,
        n_kv_els_img=n_cross_attn_layers * img_seq_len * n_kv_els_per_tok,
        n_kv_els_text_prefill=n_layers * prefill_length * n_kv_els_per_tok,
        n_kv_els_generation=n_layers * avg_seq_len * n_kv_els_per_tok,
        mlp_flops_prefill=2
        * prefill_length
        * (n_attn_layers * 13 + n_cross_attn_layers * 12.5)
        * t_config.hidden_size**2
        + 2 * img_seq_len * n_cross_attn_layers * 0.5 * t_config.hidden_size**2
        + prefill_length * embed_params,
        mlp_flops_generation=2
        * (n_attn_layers * 13 + n_cross_attn_layers * 12.5)
        * t_config.hidden_size**2
        + embed_params,
        attn_flops_prefill=prefill_length**2 * attn_flops_per_tok,
        attn_flops_generation=avg_seq_len * attn_flops_per_tok,
        cross_attn_flops_prefill=prefill_length * cross_attn_flops,
        cross_attn_flops_generation=cross_attn_flops,
    )

In [5]:
model_path = "meta-llama/Llama-3.2-11B-Vision"
config = AutoConfig.from_pretrained(model_path)

In [6]:
o1 = vision_perf(config, n_tiles=1)
print(o1)
o2 = text_perf(config, prefill_length=10, n_generated=30, img_seq_len=o1["seq_len"])
print(o2)

{'params': 818641920, 'conv_flops': 1541406720, 'mlp_flops': 1675037245440, 'attn_flops': 214748364800, 'seq_len': 1024}
{'params': 9774825472, 'n_kv_els_img': 16777216, 'n_kv_els_text_prefill': 819200, 'n_kv_els_generation': 2048000.0, 'mlp_flops_prefill': 321086554112.0, 'mlp_flops_generation': 18364760064.0, 'attn_flops_prefill': 52428800, 'attn_flops_generation': 13107200.0, 'cross_attn_flops_prefill': 1342177280, 'cross_attn_flops_generation': 134217728}
